# Docker: Part 1
## Fundamentals: Images, Containers & Dockerfiles

**Topics covered:**
1. Why Docker Exists
2. Images vs Containers
3. Core Commands
4. Writing a Dockerfile
5. Layers & Build Caching
6. Running Your FastAPI App in a Container
7. Environment Variables & Cleanup

---
> **Context:** In system design you scaled by running many identical copies of a service. Docker is how you guarantee they are identical.

## 1. Why Docker Exists

**Problem:** "It works on my machine." Different Python version, missing library, different OS: the app breaks somewhere else.

**Analogy: the shipping container.** Ports don't care what's inside the box. They only know how to move boxes. Docker packs your app + Python + libraries + config into one standard box that runs the same on any machine with Docker.

### Container vs Virtual Machine

```
VM:         App | Libs | Guest OS (GBs)  | Hypervisor | Host OS | Hardware
Container:  App | Libs                   | Docker     | Host OS | Hardware
```

A VM boots a whole operating system. A container **shares the host kernel** and only isolates the process, so it is MBs in size and starts in about a second.

### The vocabulary

| Term | Analogy | Meaning |
|---|---|---|
| **Image** | Frozen recipe | Read-only package (OS layer + Python + code + deps) |
| **Container** | A dish cooked from it | A running (or stopped) instance of an image |
| **Dockerfile** | Written recipe steps | Text file that builds an image |
| **Registry** | App store | Where images live (Docker Hub) |
| **Daemon** | The kitchen | Background service that does the work. `docker` CLI just sends it requests |

---
## 2. Images vs Containers (hands-on)

One image can produce many containers. Run the cells below and watch the IDs.

In [1]:
# Make sure the Docker daemon is up (Docker Desktop must be running)
!docker version --format 'client={{.Client.Version}} server={{.Server.Version}}'

client={.Client.Version} server={.Server.Version}


In [2]:
# Run a one-shot container, then look at the image and the container separately
!docker run --rm alpine echo "hello from a container"
!docker images alpine

hello from a container
                                                            i Info →   U  In Use
IMAGE           ID             DISK USAGE   CONTENT SIZE   EXTRA
alpine:latest   294b683cb724       13.5MB         4.19MB        


`--rm` deletes the container as soon as it exits. Without it, the stopped container stays around in `docker ps -a`

**Exercise:** run the cell above twice. Is the image pulled again? Is there a leftover container?

---
## 3. Core Commands

| Command | What it does |
|---|---|
| `docker run IMAGE` | Create + start a container from an image |
| `docker run -d` | Detached: run in the background |
| `docker run -p 8080:80` | Map host port 8080 to container port 80 |
| `docker run --name x` | Give the container a name |
| `docker run -e KEY=val` | Set an environment variable |
| `docker ps` / `docker ps -a` | Running / all containers |
| `docker logs x` | See the container's output |
| `docker exec -it x sh` | Open a shell inside a running container |
| `docker stop x` / `docker rm x` | Stop / delete a container |
| `docker images` / `docker rmi` | List / delete images |

### Ports: why `-p` exists
A container has its **own network**. Its port 80 is not your Mac's port 80 unless you publish it:

```
Browser -> localhost:8080 (your Mac) --[-p 8080:80]--> container:80
            HOST_PORT : CONTAINER_PORT
```

In [3]:
# A real long-running container: nginx web server in the background
!docker run -d --name web -p 8080:80 nginx:alpine
!docker ps --filter name=web

Unable to find image 'nginx:alpine' locally
alpine: Pulling from library/nginx

c2485f45: Pulling fs layer 
76146c12: Pulling fs layer 
389642fa: Pulling fs layer 
2a9910f5: Pulling fs layer 
3311d5df: Pulling fs layer 
d06974fa: Pulling fs layer 
Digest: sha256:df221db836e1754089190208cee7eeda94f233197056426eda74a43ab1abeac2
Status: Downloaded newer image for nginx:alpine
a60e430157ba2c0bc1ef77f93794e581ed12ffd19739854e4fe2343b728a88be
CONTAINER ID   IMAGE          COMMAND                  CREATED        STATUS                  PORTS                                     NAMES
a60e430157ba   nginx:alpine   "/docker-entrypoint.…"   1 second ago   Up Less than a second   0.0.0.0:8080->80/tcp, [::]:8080->80/tcp   web


In [4]:
# Visit it, look at its logs, peek inside, then tear it down
!curl -s localhost:8080 | head -5
!docker logs web | tail -3
!docker exec web ls /usr/share/nginx/html
!docker stop web && docker rm web

<!DOCTYPE html>
<html>
<head>
<title>Welcome to nginx!</title>
<style>
2026/10/14 08:55:13 [notice] 1#1: using the "epoll" event method
2026/10/14 08:55:13 [notice] 1#1: nginx/1.31.6
2026/10/14 08:55:13 [notice] 1#1: built by gcc 15.2.0 (Alpine 15.2.0) 
2026/10/14 08:55:13 [notice] 1#1: OS: Linux 6.12.54-linuxkit
2026/10/14 08:55:13 [notice] 1#1: getrlimit(RLIMIT_NOFILE): 1048576:1048576
2026/10/14 08:55:13 [notice] 1#1: start worker processes
2026/10/14 08:55:13 [notice] 1#1: start worker process 30
2026/10/14 08:55:13 [notice] 1#1: start worker process 31
2026/10/14 08:55:13 [notice] 1#1: start worker process 32
2026/10/14 08:55:13 [notice] 1#1: start worker process 33
2026/10/14 08:55:13 [notice] 1#1: start worker process 34
2026/10/14 08:55:13 [notice] 1#1: start worker process 35
2026/10/14 08:55:13 [notice] 1#1: start worker process 36
2026/10/14 08:55:13 [notice] 1#1: start worker process 37
2026/10/14 08:55:13 [notice] 1#1: start worker process 38
2026/10/14 08:55:13 [notice] 1

Notice the lifecycle: **run (create+start) -> running -> stop -> exited -> rm (gone)**. A container only lives as long as its main process.

---
## 4. Writing a Dockerfile

Running other people's images is half the story. A **Dockerfile** builds yours. Open `../hello_api/Dockerfile`:

```dockerfile
FROM python:3.11-slim            # base image: Linux + Python
WORKDIR /app                     # cd into /app (created if missing)
COPY requirements.txt .          # copy file from your Mac into the image
RUN pip install -r requirements.txt   # runs at BUILD time
COPY . .                         # copy the rest of the code
EXPOSE 8000                      # documentation only: app listens on 8000
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]   # runs at START time
```

### RUN vs CMD (a classic confusion)
- `RUN` executes **while building** the image (installing things). Result is baked in.
- `CMD` is the **default command when a container starts**. Only one, and it can be overridden.

### Why `--host 0.0.0.0`?
Inside a container, `127.0.0.1` means "only this container". Your port mapping arrives from outside, so the app must listen on **all interfaces** (`0.0.0.0`) or you will never reach it. This is the #1 beginner bug.

## RUN vs CMD, in simple terms

Think of **building a food truck** and then **opening it for customers**.

- **`RUN` = setting up the truck (once, at build time).** You buy the stove, stock the fridge, install the shelves. When you're done, everything stays inside the truck.
- **`CMD` = "open for business" (every time the truck starts).** Each morning you turn the key and start cooking.

```dockerfile
RUN pip install -r requirements.txt     # setup: install packages ONCE while building
CMD ["uvicorn", "main:app"]             # start: launch the app EVERY time a container starts
```

| | `RUN` | `CMD` |
|---|---|---|
| When | While building the image | When a container starts |
| Purpose | Prepare the environment | Start the app |
| How many | As many as you need | Only one (the last one counts) |
| Result | Saved inside the image | Not saved, it's just "what to run" |

**Why it matters:**
- `pip install` in `CMD` would reinstall everything every time a container starts, which is slow.
- `uvicorn` in `RUN` would try to start the server during the build, and the build would never finish.

"Can be overridden" means you can replace the start command at run time:

```bash
docker run hello-api:1.0 echo "hi"   # runs echo instead of uvicorn
```

---

## Why `--host 0.0.0.0`?

Think of **an apartment building with a front gate**.

- **`127.0.0.1`** means "only talk to people inside my own room". The app listens, but only to itself.
- **`0.0.0.0`** means "listen to anyone who knocks, from any door".

A container is like its own sealed room. When you visit `localhost:8000`, Docker's port mapping carries the request **from outside into the container**. To the app, it looks like a visitor from outside the room.

```
Browser --> your Mac:8000 --> [Docker port mapping] --> container
```

- App listens on `127.0.0.1`: it only accepts visitors from inside the container, so your request is ignored ("connection refused" or an empty reply).
- App listens on `0.0.0.0`: it accepts the visitor coming through the port mapping, and it works.

On your Mac outside Docker, `127.0.0.1` is fine, which is why this bug only appears once you containerize.

> **Symptom to remember:** the container shows as running and the logs look healthy, but the page won't load. Check the host setting first.

**One-line summary:** `RUN` prepares the image once, `CMD` starts the app each time, and `0.0.0.0` tells the app to accept connections from outside its container.


---
## 5. Layers & Build Caching

Each instruction creates a **layer**. Docker caches layers and reuses them if nothing above them changed.

```
FROM python:3.11-slim        layer 1  (cached forever)
COPY requirements.txt .      layer 2  (changes rarely)
RUN pip install ...          layer 3  (slow! reused while layer 2 unchanged)
COPY . .                     layer 4  (changes on every code edit)
```

That is why we copy `requirements.txt` **before** the code. If we did `COPY . .` first, every one-line code edit would invalidate the cache and reinstall every package.

**`.dockerignore`** works like `.gitignore`: keeps `.git`, `__pycache__`, and `.env` secrets out of the image.

In [5]:
# Build the image from the Dockerfile. -t gives it a name:tag. The last "." is the build context.
!docker build -t hello-api:1.0 ../hello_api

[+] Building 0.0s (0/1)                                    docker:desktop-linux
[+] Building 0.2s (1/2)                                    docker:desktop-linux
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 531B                                       0.0s
 => [internal] load metadata for docker.io/library/python:3.11-slim        0.2s
[+] Building 0.3s (1/2)                                    docker:desktop-linux
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 531B                                       0.0s
 => [internal] load metadata for docker.io/library/python:3.11-slim        0.3s
[+] Building 0.5s (1/2)                                    docker:desktop-linux
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 531B                                       0.0s
 => [internal] load metadata for docker.

In [6]:
# Build again with NO changes: every step should say CACHED
!docker build -t hello-api:1.0 ../hello_api

[+] Building 0.0s (0/1)                                    docker:desktop-linux
[+] Building 0.2s (1/2)                                    docker:desktop-linux
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 531B                                       0.0s
 => [internal] load metadata for docker.io/library/python:3.11-slim        0.2s
[+] Building 0.3s (1/2)                                    docker:desktop-linux
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 531B                                       0.0s
 => [internal] load metadata for docker.io/library/python:3.11-slim        0.3s
[+] Building 0.5s (1/2)                                    docker:desktop-linux
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 531B                                       0.0s
 => [internal] load metadata for docker.

In [7]:
# See the layers and the size
!docker history hello-api:1.0
!docker images hello-api

IMAGE          CREATED          CREATED BY                                      SIZE      COMMENT
077f27171ca4   3 seconds ago    CMD ["uvicorn" "main:app" "--host" "0.0.0.0"…   0B        buildkit.dockerfile.v0
<missing>      3 seconds ago    EXPOSE [8000/tcp]                               0B        buildkit.dockerfile.v0
<missing>      3 seconds ago    COPY . . # buildkit                             20.5kB    buildkit.dockerfile.v0
<missing>      3 seconds ago    RUN /bin/sh -c pip install --no-cache-dir -r…   54.9MB    buildkit.dockerfile.v0
<missing>      21 seconds ago   COPY requirements.txt . # buildkit              12.3kB    buildkit.dockerfile.v0
<missing>      21 seconds ago   WORKDIR /app                                    8.19kB    buildkit.dockerfile.v0
<missing>      8 days ago       CMD ["python3"]                                 0B        buildkit.dockerfile.v0
<missing>      8 days ago       RUN /bin/sh -c set -eux;  for src in idle3 p…   16.4kB    buildkit.dockerfile.v

---
## 6. Running Your App in a Container

In [8]:
!docker run -d --name api -p 8000:8000 hello-api:1.0
!sleep 2 && curl -s localhost:8000/

f7799fc4c6dcb638c19dae60db162629c7c810ede8cf2b2825786c49d045254b
{"message":"Hello from inside a container!"}

In [9]:
# The hostname inside a container is its container ID
!curl -s localhost:8000/whoami
!docker ps --filter name=api --format '{{.ID}}  {{.Names}}  {{.Ports}}'

{"hostname":"f7799fc4c6dc","greeting":"hi"}{.ID}  {.Names}  {.Ports}


---
## 7. Environment Variables & Cleanup

Config (API keys, DB URLs, greetings) should come from the **environment**, not be baked into the image. The same image then runs in dev, staging, and prod with different settings.

In [10]:
# Same image, different config, second container on a different host port
!docker run -d --name api2 -p 8001:8000 -e GREETING="namaste" hello-api:1.0
!sleep 2 && curl -s localhost:8001/whoami

a6241899808af388af3d2ce557403b2f94d1edc9d5f607589f4c0ca93730da4f
{"hostname":"a6241899808a","greeting":"namaste"}

This is horizontal scaling in miniature: **two containers from one image**, different IDs, different config. Put a load balancer in front and you have the system design diagram from Part 1.

In [11]:
# Cleanup: stop and remove both containers
!docker rm -f api api2
!docker ps -a --filter name=api

api
api2
CONTAINER ID   IMAGE     COMMAND   CREATED   STATUS    PORTS     NAMES


# Docker Commands Cheat Sheet

## The general layout

```
docker  [object]  [action]  [options]  [target]  [arguments]
  │        │         │          │          │          │
  │        │         │          │          │          └─ command to run, etc.
  │        │         │          │          └─ image / container name or ID
  │        │         │          └─ flags like -d, -p, --name
  │        │         └─ run, ls, rm, stop...
  │        └─ container, image, volume, network
  └─ the CLI
```

Short and long forms do the same thing:

```
docker ps         ==  docker container ls
docker images     ==  docker image ls
docker rm web     ==  docker container rm web
docker rmi nginx  ==  docker image rm nginx
```

Options go before the image name. Anything after the image is passed to the program inside:

```
docker run  -d -p 8080:80  nginx:alpine  nginx -g "daemon off;"
            └─ options ─┘  └─ image ──┘  └─ command inside ──┘
```

## Essential commands

| Command layout | What it does | Example |
|---|---|---|
| `docker run [options] IMAGE [command]` | Creates and starts a container | `docker run -d --name web -p 8080:80 nginx:alpine` |
| `docker run -e KEY=value IMAGE` | Passes an environment variable | `docker run -d -e GREETING=namaste hello-api:1.0` |
| `docker run --rm IMAGE [command]` | Runs once, then auto-deletes the container | `docker run --rm alpine echo hi` |
| `docker run -it IMAGE bash` | Opens an interactive shell in a new container | `docker run -it ubuntu bash` |
| `docker run -v VOLUME:PATH IMAGE` | Persists data in a volume | `docker run -v mydata:/var/lib/postgresql/data postgres:16` |
| `docker ps [-a]` | Lists running containers (`-a` for all) | `docker ps -a` |
| `docker logs [-f] NAME` | Shows container logs (`-f` follows live) | `docker logs -f --tail 20 web` |
| `docker exec -it NAME CMD` | Runs a command inside a running container | `docker exec -it web sh` |
| `docker stop NAME` / `docker start NAME` | Stops or starts a container | `docker stop web` |
| `docker rm [-f] NAME` | Removes a container (`-f` forces) | `docker rm -f web` |
| `docker images` | Lists local images | `docker images` |
| `docker build -t NAME:TAG PATH` | Builds an image from a Dockerfile | `docker build -t hello-api:1.0 .` |
| `docker pull IMAGE:TAG` / `docker push IMAGE` | Downloads or uploads an image | `docker pull postgres:16-alpine` |
| `docker rmi IMAGE` | Deletes an image | `docker rmi hello-api:1.0` |
| `docker network create NAME` | Creates a network for containers to talk | `docker network create mynet` |
| `docker volume create NAME` | Creates a volume for persistent data | `docker volume create pgdata` |
| `docker system prune` | Removes stopped containers, unused networks, dangling images | `docker system prune` |

## Docker Compose (multi-container)

| Command layout | What it does | Example |
|---|---|---|
| `docker compose up [-d]` | Starts all services (`-d` for background) | `docker compose up -d` |
| `docker compose down` | Stops and removes the services | `docker compose down` |
| `docker compose ps` | Lists the services | `docker compose ps` |
| `docker compose logs [-f] [service]` | Shows service logs | `docker compose logs -f redis` |
| `docker compose exec SERVICE CMD` | Runs a command inside a service | `docker compose exec postgres psql -U postgres` |

## Worked example

```
docker build -t hello-api:1.0 .                          # 1. build image
docker run -d --name api -p 8000:8000 hello-api:1.0      # 2. run container
docker ps                                                # 3. check it's up
docker logs -f api                                       # 4. watch logs (Ctrl+C to leave)
docker exec -it api sh                                   # 5. look inside (exit to leave)
docker stop api && docker rm api                         # 6. stop and remove
docker rmi hello-api:1.0                                 # 7. delete the image
```

## Memory tricks

- `-d` = detached, `-p` = port, `-e` = environment, `-v` = volume, `-it` = interactive terminal.
- `-a` usually means "all". `-f` means "force" or "follow" depending on the command.
- A container name or the first 3-4 characters of its ID both work as a target.
- Five commands cover about 80% of daily use: `run`, `ps`, `logs`, `exec`, `stop`/`rm`.